In [2]:
!pip install Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.0 MB/s eta 0:00:00


In [3]:
# Importamos la libreria de colab para poder usar el key_vault , si se ejecuta en local se puede optar por usar un .env
from google.colab import userdata
api_key = userdata.get('GROQ_API_KEY')
SERAPI = userdata.get('SER_API_KEY')
 # Importamos la libreria para comunicarnos con la api
from groq import Groq

#Creamos la conexion con la funcion Groq
client = Groq(api_key=api_key)


In [4]:
# El scrapping se ejecuta en 2 etapas
# 1) Pasamos la busqueda a la web https://serpapi.com/search
# 2) Descargamos los datos en memoria mediante requests

import requests

def buscador_con_serpapi(query:str)->str:
  """
  Función para realizar una busqueda en internet y extraer el resultado.
  in:
    query: str
  out:
    url: str
    title: str
    text: str
  """

  #URL de la pagina serpapi adonde vamos a enviar la consulta
  url = "https://serpapi.com/search"

  # Parametros necesarios: cual es la query, apikey que generamos en la pagina de serpapi, y el motor de consulta que debera usar
  params = {
        "q": query,
        "api_key": SERAPI,
        "engine": "google"
  }

  # Descargamos la información generada por la pagina serpapi
  response = requests.get(url, params=params)
  data = response.json()


  # La información devuelta tendra formato JSON y se debera consultar en la documentación de SERPAPI cuales son los key disponibles
  r = data["organic_results"][0]

  # Tomamos los valores de titulo,link y texto/cuerpo para el primer resultado del tipo ORGANICO
  title = r["title"]
  link = r["link"]
  text = r.get("snippet", "")

  return (url, title, text)


In [6]:
# Preparamos una consulta
query = "Quien fue el ganador, y por cuanto gano, del partido de futbol de Argentina contra Honduras en Junio del 26"

# Llamamos a la funcion y obtenemos los resultados
url, title, text = buscador_con_serpapi(query)


In [7]:
# Formateamos el contexto pasando como argumento los datos obtenidos
contexto = f"""
FUENTE:
Título: {title}
URL: {url}
Contenido: {text[:1000]}
"""


# Realizamos la misma pregunta que probamos anteriormente
pregunta = "Quien fue el ganador, y por cuanto gano, del partido de futbol de Argentina contra Honduras en Junio del 26"

# Ejecutamos la llamada al LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": f"""
Usa el siguiente contexto para responder:

{contexto}

PREGUNTA:
{pregunta}
"""
        }
    ]
)

print(response.choices[0].message.content)


La Selección Argentina fue el ganador del partido. El resultado fue de 2 a 0 en su favor. El partido se disputó en el Kyle Field en junio del 26 (en este caso no se especifica la fecha exacta, sin embargo se menciona que se trata de la fecha "del 26" y sabiendo que "del 26" es en junio y se sabe que el partido se trata del "Junio del 26")
